In [0]:
CREATE OR REFRESH STREAMING TABLE circuitbox.bronze.orders
  COMMENT 'Bronze data for customers'
  TBLPROPERTIES ('quality' = 'bronze') AS
select
  *,
  _metadata.file_path as file_path,
  current_timestamp as ingestion_timestamp
from
  cloud_files(
    '/Volumes/circuitbox/landing/operational_data/orders/',
    'json',
    map(
      "cloudFiles.inferColumnTypes",
      "true",
      'cloudFiles.schemaLocation',
      '/Volumes/circuitbox/landing/operational_data/orders/schema'
    )
  );

In [0]:
CREATE OR REFRESH STREAMING TABLE circuitbox.lakehouse.orders_clean (

    CONSTRAINT valid_customer_id EXPECT(customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
    CONSTRAINT valid_order_id EXPECT(order_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
    CONSTRAINT valid_order_status EXPECT(
      order_status IN ('Pending', 'Completed', 'Shipped', 'Cancelled')
    ),
    CONSTRAINT valid_payment_method EXPECT(
      payment_method IN ('Bank Transfer', 'PayPal', 'Credit Card')
    )
  )
  COMMENT 'Staging data for customers'
  TBLPROPERTIES ('quality' = 'staging') AS
select
  order_id,
  customer_id,
  CAST(order_timestamp as TIMESTAMP) AS order_timestamp,
  payment_method,
  items,
  order_status
from
  STREAM(LIVE.circuitbox.bronze.orders)

In [0]:
CREATE OR REFRESH STREAMING TABLE circuitbox.silver.orders
  COMMENT 'Silver data for customers'
  TBLPROPERTIES ('quality' = 'silver') AS
select order_id,
        customer_id,
        order_timestamp,
        payment_method,
        order_status,
        item.item_id,
        item.name as item_name,
        item.category as item_category,
        item.price as item_price,
        item.quantity as item_quantity
from (
  SELECT order_id,
    customer_id,
    order_timestamp,
    payment_method,
    order_status,
    explode(items) AS item
 FROM STREAM(LIVE.circuitbox.lakehouse.orders_clean)
)